# Smoke Test — April 9 + Both April 20 Records

Runs all three record trainers back-to-back on a single Colab GPU using the same data.

**What each cell does:**
1. GPU check
2. Clone / update repo
3. Install repo + notebook dependencies
4. Select a smoke profile (`compare` by default, optional `t4`)
5. Download shared data and build a profile-specific capped smoke-val split (used by all three models)
6. Train + export: April 9 `LegalTTT`
7. Train + export: April 20 `AttnGate_MultiPhaseTTT_LaCT`
8. Train + export: April 20 `3LayerRecur_ParResid_AttnGate_MP4TTT`
9. Compare final metrics from all three logs

**Profiles:**
- `compare` (default): `ITERATIONS=300`, `TRAIN_BATCH_TOKENS=32768`, `VAL_BATCH_TOKENS=32768`, `SMOKE_VAL_MAX_TOKENS=4194304`. Use this when you want the stronger apples-to-apples comparison run.
- `t4`: `ITERATIONS=150`, `TRAIN_BATCH_TOKENS=16384`, `VAL_BATCH_TOKENS=16384`, `SMOKE_VAL_MAX_TOKENS=1048576`. Use this when you need a materially cheaper smoke on a Tesla T4.

**Common settings (all three models share these):**
- `MAX_WALLCLOCK_SECONDS=0` — no time cap; let the selected profile finish all configured steps
- `VAL_LOSS_EVERY=0` — skip periodic val; final only
- `GPTQ_CALIBRATION_BATCHES=8` — fast export check
- `PG_COLAB_DISABLE_COMPILE=1` — disables `torch.compile` via `sitecustomize.py`
- No FlashAttention source build in this notebook; the local `flash_attn_interface.py` stub is used where needed

**Why the default `compare` cap is `4194304` tokens:** this is `2048` full eval sequences of length `2048` and `128` legal-TTT chunks of `32768` tokens, so each model still gets millions of scored tokens and a meaningful adaptation schedule instead of just a handful of updates. It is also about `10.3%` of the April 9 full validation load (`40540160` tokens), which should cut sliding-window and TTT smoke-eval time by roughly `9.7x` while keeping the three smoke runs directly comparable to each other.

**Why the `t4` cap is `1048576` tokens:** this is still `512` full eval sequences and `32` legal-TTT chunks, which is enough for a path-check smoke, but is much cheaper on a T4 than the stronger comparison profile.

**OOM recovery:** lower `GPTQ_CALIBRATION_BATCHES=4` or `TRAIN_BATCH_TOKENS=16384` in the failing cell only.

In [ ]:
%%bash
nvidia-smi

In [9]:
%%bash
set -euo pipefail
REPO_DIR="$(pwd)/parameter-golf"
if [[ ! -d "$REPO_DIR/.git" ]]; then
  git clone https://github.com/IanniMuliterno/parameter-golf.git "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin
  git -C "$REPO_DIR" pull --ff-only
fi
echo "commit: $(git -C "$REPO_DIR" rev-parse --short HEAD)"

Updating a227f65..18e5ebc
Fast-forward
 .../README.md                                      |  393 ++++
 .../flash_attn_interface.py                        |   47 +
 .../requirements.txt                               |    3 +
 .../run.sh                                         |   96 +
 .../train_gpt.py                                   | 2053 ++++++++++++++++++++
 5 files changed, 2592 insertions(+)
 create mode 100644 records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/README.md
 create mode 100644 records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/flash_attn_interface.py
 create mode 100644 records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/requirements.txt
 create mode 100755 records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/run.sh
 create mode 100644 records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/train_gpt.py
commit: 18e5ebc


From https://github.com/IanniMuliterno/parameter-golf
   a227f65..18e5ebc  main       -> origin/main


In [3]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
python3 -m pip install -r "$REPO/requirements.txt" -r "$REPO/colab/2026-04-23_SmokeTest_Records/requirements.txt"
python3 - <<'PY'
import brotli
import huggingface_hub
import sentencepiece
print('Notebook dependencies ready.')
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 5.5 MB/s eta 0:00:00
Notebook dependencies ready.


In [5]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
CONFIG_PATH="$REPO/colab/2026-04-23_SmokeTest_Records/smoke_profile.env"
PROFILE="t4"  # change to "t4" for the lighter Tesla T4 smoke profile

case "$PROFILE" in
  compare)
    cat > "$CONFIG_PATH" <<'EOF'
export SMOKE_PROFILE=compare
export SMOKE_RUN_PREFIX=smoke
export SMOKE_DATA_DIR_NAME=data_smoke_compare
export SMOKE_VAL_MAX_TOKENS=4194304
export SMOKE_ITERATIONS=300
export SMOKE_TRAIN_BATCH_TOKENS=32768
export SMOKE_VAL_BATCH_TOKENS=32768
export SMOKE_GPTQ_CALIBRATION_BATCHES=8
EOF
    ;;
  t4)
    cat > "$CONFIG_PATH" <<'EOF'
export SMOKE_PROFILE=t4
export SMOKE_RUN_PREFIX=smoke_t4
export SMOKE_DATA_DIR_NAME=data_smoke_t4
export SMOKE_VAL_MAX_TOKENS=1048576
export SMOKE_ITERATIONS=150
export SMOKE_TRAIN_BATCH_TOKENS=16384
export SMOKE_VAL_BATCH_TOKENS=16384
export SMOKE_GPTQ_CALIBRATION_BATCHES=8
EOF
    ;;
  *)
    echo "Unsupported PROFILE=$PROFILE. Use compare or t4." >&2
    exit 1
    ;;
esac

echo "Active smoke profile config:"
cat "$CONFIG_PATH"

Active smoke profile config:
export SMOKE_PROFILE=t4
export SMOKE_RUN_PREFIX=smoke_t4
export SMOKE_DATA_DIR_NAME=data_smoke_t4
export SMOKE_VAL_MAX_TOKENS=1048576
export SMOKE_ITERATIONS=150
export SMOKE_TRAIN_BATCH_TOKENS=16384
export SMOKE_VAL_BATCH_TOKENS=16384
export SMOKE_GPTQ_CALIBRATION_BATCHES=8


## Set latency relevant variables according to `SMOKE_PROFILE`

In [6]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
CONFIG_PATH="$REPO/colab/2026-04-23_SmokeTest_Records/smoke_profile.env"
[[ -f "$CONFIG_PATH" ]] || { echo "Run the smoke profile cell first." >&2; exit 1; }
source "$CONFIG_PATH"
export SMOKE_PROFILE SMOKE_RUN_PREFIX SMOKE_DATA_DIR_NAME SMOKE_VAL_MAX_TOKENS
# Download 5 train shards + full val, then build one explicit profile-specific smoke-val root.
# sp8192 dataset lives on kevclark/parameter-golf, not the default repo.
MATCHED_FINEWEB_REPO_ID=kevclark/parameter-golf \
  python3 "$REPO/data/cached_challenge_fineweb.py" --variant sp8192 --train-shards 5
python3 - <<'PY'
import os
import shutil
from pathlib import Path

import numpy as np

HEADER_INTS = 256
HEADER_BYTES = HEADER_INTS * np.dtype('<i4').itemsize
TOKEN_DTYPE = np.dtype('<u2')
EXPECTED_MAGIC = 20240520
EXPECTED_VERSION = 1
SMOKE_PROFILE = os.environ['SMOKE_PROFILE']
SMOKE_VAL_MAX_TOKENS = int(os.environ['SMOKE_VAL_MAX_TOKENS'])
EVAL_SEQ_LEN = 2048
TTT_CHUNK_TOKENS = 32_768

repo = Path('parameter-golf')
source_dataset_dir = repo / 'data' / 'datasets' / 'fineweb10B_sp8192'
source_tokenizer_dir = repo / 'data' / 'tokenizers'
smoke_data_dir = repo / os.environ['SMOKE_DATA_DIR_NAME']
smoke_dataset_dir = smoke_data_dir / 'datasets' / 'fineweb10B_sp8192'
smoke_tokenizer_dir = smoke_data_dir / 'tokenizers'
train_files = sorted(source_dataset_dir.glob('fineweb_train_*.bin'))
val_files = sorted(source_dataset_dir.glob('fineweb_val_*.bin'))
if len(train_files) < 5:
    raise SystemExit(f'expected at least 5 train shards in {source_dataset_dir}, found {len(train_files)}')
if not val_files:
    raise SystemExit(f'expected at least 1 val shard in {source_dataset_dir}, found 0')
if SMOKE_VAL_MAX_TOKENS % EVAL_SEQ_LEN != 0:
    raise SystemExit(
        f'SMOKE_VAL_MAX_TOKENS={SMOKE_VAL_MAX_TOKENS} must be divisible by EVAL_SEQ_LEN={EVAL_SEQ_LEN}'
    )
if SMOKE_VAL_MAX_TOKENS % TTT_CHUNK_TOKENS != 0:
    raise SystemExit(
        f'SMOKE_VAL_MAX_TOKENS={SMOKE_VAL_MAX_TOKENS} must be divisible by TTT_CHUNK_TOKENS={TTT_CHUNK_TOKENS}'
    )

def read_header(path: Path) -> np.ndarray:
    header = np.fromfile(path, dtype='<i4', count=HEADER_INTS)
    if header.size != HEADER_INTS:
        raise SystemExit(f'short header in {path}')
    if int(header[0]) != EXPECTED_MAGIC or int(header[1]) != EXPECTED_VERSION:
        raise SystemExit(f'unexpected shard header in {path}: {header[:3].tolist()}')
    return header

full_val_tokens = 0
val_parts = []
remaining = SMOKE_VAL_MAX_TOKENS
first_header = None
for path in val_files:
    header = read_header(path)
    if first_header is None:
        first_header = header.copy()
    shard_tokens = int(header[2])
    full_val_tokens += shard_tokens
    if remaining <= 0:
        continue
    take = min(remaining, shard_tokens)
    tokens = np.fromfile(path, dtype=TOKEN_DTYPE, count=take, offset=HEADER_BYTES)
    if tokens.size != take:
        raise SystemExit(f'short token read in {path}: expected {take}, found {tokens.size}')
    val_parts.append(tokens)
    remaining -= take
if first_header is None:
    raise SystemExit('missing validation header after listing val shards')
if full_val_tokens < SMOKE_VAL_MAX_TOKENS:
    raise SystemExit(
        f'full validation split has only {full_val_tokens} tokens, below requested SMOKE_VAL_MAX_TOKENS={SMOKE_VAL_MAX_TOKENS}'
    )
if remaining != 0:
    raise SystemExit(f'failed to collect capped val tokens exactly; remaining={remaining}')

if smoke_data_dir.exists():
    shutil.rmtree(smoke_data_dir)
smoke_dataset_dir.mkdir(parents=True, exist_ok=False)
smoke_tokenizer_dir.mkdir(parents=True, exist_ok=False)

for path in train_files:
    os.symlink(path.resolve(), smoke_dataset_dir / path.name)
tokenizer_files = sorted(source_tokenizer_dir.glob('fineweb_8192_bpe.*'))
if not tokenizer_files:
    raise SystemExit(f'expected tokenizer files in {source_tokenizer_dir}, found 0 matching fineweb_8192_bpe.*')
for path in tokenizer_files:
    os.symlink(path.resolve(), smoke_tokenizer_dir / path.name)

smoke_tokens = np.concatenate(val_parts)
if smoke_tokens.size != SMOKE_VAL_MAX_TOKENS:
    raise SystemExit(
        f'constructed capped validation token count mismatch: expected {SMOKE_VAL_MAX_TOKENS}, found {smoke_tokens.size}'
    )
smoke_header = first_header.copy()
smoke_header[2] = SMOKE_VAL_MAX_TOKENS
smoke_val_path = smoke_dataset_dir / 'fineweb_val_000000.bin'
with smoke_val_path.open('wb') as f:
    f.write(smoke_header.astype('<i4', copy=False).tobytes())
    f.write(smoke_tokens.astype(TOKEN_DTYPE, copy=False).tobytes())

print(
    f'Source data: profile={SMOKE_PROFILE} '
    f'{len(train_files)} train shard(s), {len(val_files)} val shard(s), full_val_tokens={full_val_tokens}'
)
print(
    'Smoke data ready: '
    f'profile={SMOKE_PROFILE} '
    f'train_shards={len(train_files)} '
    f'smoke_val_tokens={SMOKE_VAL_MAX_TOKENS} '
    f'eval_sequences={SMOKE_VAL_MAX_TOKENS // EVAL_SEQ_LEN} '
    f'ttt_chunks={SMOKE_VAL_MAX_TOKENS // TTT_CHUNK_TOKENS} '
    f'ratio={SMOKE_VAL_MAX_TOKENS / full_val_tokens:.4f} '
    f'data_dir={smoke_data_dir}'
)
PY

Source data: profile=t4 5 train shard(s), 1 val shard(s), full_val_tokens=40540803
Smoke data ready: profile=t4 train_shards=5 smoke_val_tokens=1048576 eval_sequences=512 ttt_chunks=32 ratio=0.0259 data_dir=parameter-golf/data_smoke_t4


## April 9 — `SP8192_3LayerRecur_ParResid_QK525_LegalTTT`

Baseline record. Uses legacy fixed-bits export (6-bit matrices, 8-bit embeddings).
No attention gate, no multi-phase TTT.

Expected log lines: `pre-quantization post-ema`, `quantized`, `quantized_sliding_window`, `artifact bytes`.

In [7]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
CONFIG_PATH="$COLAB_DIR/smoke_profile.env"
[[ -f "$CONFIG_PATH" ]] || { echo "Run the smoke profile cell first." >&2; exit 1; }
source "$CONFIG_PATH"
REC="$REPO/records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT"
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR="$REPO/$SMOKE_DATA_DIR_NAME" \
SEED=42 \
RUN_ID="${SMOKE_RUN_PREFIX}_apr9" \
ITERATIONS="$SMOKE_ITERATIONS" \
TRAIN_BATCH_TOKENS="$SMOKE_TRAIN_BATCH_TOKENS" \
VAL_BATCH_TOKENS="$SMOKE_VAL_BATCH_TOKENS" \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES="$SMOKE_GPTQ_CALIBRATION_BATCHES" \
GPTQ_RESERVE_SECONDS=30 \
python3 train_gpt.py

Hyperparameters:
  adam_eps: 1e-08
  adam_wd: 0.02
  beta1: 0.9
  beta2: 0.95
  compressor: brotli
  data_dir: /content/parameter-golf/data_smoke_t4
  datasets_dir: /content/parameter-golf/data_smoke_t4/datasets/fineweb10B_sp8192
  distributed: False
  ema_decay: 0.9965
  embed_bits: 8
  embed_clip_sigmas: 20.0
  embed_lr: 0.6
  embed_wd: 0.085
  embedding_dim: 512
  enable_looping_at: 0.35
  etlb_clip: 3.0
  etlb_enabled: False
  etlb_lr: 0.05
  etlb_steps: 5
  eval_seq_len: 2048
  eval_stride: 64
  gptq_calibration_batches: 8
  gptq_reserve_seconds: 30.0
  grad_accum_steps: 8
  grad_clip_norm: 0.3
  head_lr: 0.008
  is_main_process: True
  iterations: 150
  ln_scale: True
  local_rank: 0
  logfile: logs/smoke_t4_apr9.txt
  logit_softcap: 30.0
  loop_end: 5
  loop_start: 3
  matrix_bits: 6
  matrix_clip_sigmas: 12.85
  matrix_lr: 0.022
  max_wallclock_seconds: 0.0
  min_lr: 0.0
  mlp_mult: 4.0
  model_dim: 512
  model_path: final_model.pt
  muon_backend_steps: 5
  muon_beta2: 0.95
  m

## April 20 #1 — `SP8192_AttnGate_MultiPhaseTTT_LaCT`

Adds zero-init attention output gate (`ATTN_GATE_ENABLED=1`) and 4-phase score-first TTT
(`MULTIPHASE_TTT_ENABLED=1`). LaCT adapter is disabled here to keep the run fast.
Uses the entropy-constrained GPTQ allocator.

Expected log lines: `pre-quantization post-ema`, `quantized`, `quantized_sliding_window`,
`allocator_selected`, `artifact bytes`.

In [10]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
CONFIG_PATH="$COLAB_DIR/smoke_profile.env"
[[ -f "$CONFIG_PATH" ]] || { echo "Run the smoke profile cell first." >&2; exit 1; }
source "$CONFIG_PATH"
REC="$REPO/records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT"
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR="$REPO/$SMOKE_DATA_DIR_NAME" \
SEED=42 \
RUN_ID="${SMOKE_RUN_PREFIX}_apr20_attngate_mpttt" \
ITERATIONS="$SMOKE_ITERATIONS" \
TRAIN_BATCH_TOKENS="$SMOKE_TRAIN_BATCH_TOKENS" \
VAL_BATCH_TOKENS="$SMOKE_VAL_BATCH_TOKENS" \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES="$SMOKE_GPTQ_CALIBRATION_BATCHES" \
GPTQ_RESERVE_SECONDS=30 \
python3 train_gpt.py

Hyperparameters:
  adam_eps: 1e-08
  adam_wd: 0.02
  allocator_attn_bits: (6, 7)
  allocator_attn_sigmas: None
  allocator_code_wrappers: ('source', 'lzma_raw_b85_exec')
  allocator_embed_bits: (8,)
  allocator_embed_sigmas: (16.0, 20.0, 24.0)
  allocator_group_cols: 128
  allocator_lambdas: (0.0, 1e-09, 3e-09, 1e-08, 3e-08, 1e-07, 3e-07, 1e-06, 3e-06, 1e-05)
  allocator_matrix_bits: (5, 6, 7)
  allocator_matrix_sigmas: (10.5, 12.85, 15.0)
  allocator_mlp_bits: None
  allocator_mlp_sigmas: None
  allocator_use_entropy_proxy: True
  artifact_target_bytes: 16000000
  attn_gate_enabled: True
  beta1: 0.9
  beta2: 0.95
  compressor: brotli
  data_dir: /content/parameter-golf/data_smoke_t4
  datasets_dir: /content/parameter-golf/data_smoke_t4/datasets/fineweb10B_sp8192
  distributed: False
  ema_decay: 0.9965
  embed_bits: 8
  embed_clip_sigmas: 20.0
  embed_lr: 0.6
  embed_wd: 0.085
  embedding_dim: 512
  enable_looping_at: 0.35
  eval_seq_len: 2048
  eval_stride: 64
  export_allocator: en

## April 20 #2 — `SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT`

Combines 3-layer recurrence + parallel residual with attention output gate
(`ATTN_OUT_GATE_ENABLED=1`, note different flag name from #1) and 4-phase TTT
(`TTT_PHASES=4`, the default). Has built-in torch SDPA fallback — no flash-attn needed.
Uses the entropy-constrained GPTQ allocator.

Expected log lines: same as April 20 #1.

In [11]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
CONFIG_PATH="$COLAB_DIR/smoke_profile.env"
[[ -f "$CONFIG_PATH" ]] || { echo "Run the smoke profile cell first." >&2; exit 1; }
source "$CONFIG_PATH"
REC="$REPO/records/track_10min_16mb/2026-04-20_SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT"
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR="$REPO/$SMOKE_DATA_DIR_NAME" \
SEED=42 \
RUN_ID="${SMOKE_RUN_PREFIX}_apr20_mp4ttt" \
ITERATIONS="$SMOKE_ITERATIONS" \
TRAIN_BATCH_TOKENS="$SMOKE_TRAIN_BATCH_TOKENS" \
VAL_BATCH_TOKENS="$SMOKE_VAL_BATCH_TOKENS" \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES="$SMOKE_GPTQ_CALIBRATION_BATCHES" \
GPTQ_RESERVE_SECONDS=30 \
python3 train_gpt.py

Hyperparameters:
  adam_eps: 1e-08
  adam_wd: 0.02
  attention_backend: flash_attn_interface
  attn_out_gate_enabled: True
  awq_powers: (0.0, 0.25, 0.5)
  beta1: 0.9
  beta2: 0.95
  compressor: brotli
  data_dir: /content/parameter-golf/data_smoke_t4
  datasets_dir: /content/parameter-golf/data_smoke_t4/datasets/fineweb10B_sp8192
  distributed: False
  ema_decay: 0.9965
  embed_bits: 8
  embed_clip_sigmas: (20.0,)
  embed_lr: 0.6
  embed_wd: 0.085
  embedding_dim: 512
  enable_looping_at: 0.35
  etlb_clip: 3.0
  etlb_enabled: False
  etlb_lr: 0.05
  etlb_steps: 5
  eval_seq_len: 2048
  eval_stride: 64
  gptq_accept_mse_ratio: 1.05
  gptq_calibration_batches: 8
  gptq_reserve_seconds: 30.0
  grad_accum_steps: 8
  grad_clip_norm: 0.3
  head_lr: 0.008
  hessian_damping: 0.01
  is_main_process: True
  iterations: 150
  ln_scale: True
  local_rank: 0
  logfile: logs/smoke_t4_apr20_mp4ttt.txt
  logit_softcap: 30.0
  loop_end: 5
  loop_start: 3
  matrix_bits: 6
  matrix_clip_sigmas: (12.85, 

## Compare final metrics from the active profile

Run this after all three training cells finish. `val_tokens` should match the active profile cap in every log.

In [12]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
CONFIG_PATH="$COLAB_DIR/smoke_profile.env"
[[ -f "$CONFIG_PATH" ]] || { echo "Run the smoke profile cell first." >&2; exit 1; }
source "$CONFIG_PATH"
APR9="$REPO/records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT/logs/${SMOKE_RUN_PREFIX}_apr9.txt"
APR20A="$REPO/records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/logs/${SMOKE_RUN_PREFIX}_apr20_attngate_mpttt.txt"
APR20B="$REPO/records/track_10min_16mb/2026-04-20_SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT/logs/${SMOKE_RUN_PREFIX}_apr20_mp4ttt.txt"

echo "Active profile: $SMOKE_PROFILE"
for LOG in "$APR9" "$APR20A" "$APR20B"; do
  echo "=== $(basename $(dirname $LOG))/$(basename $LOG) ==="
  grep -E "val_tokens:|pre-quantization post-ema|quantized|artifact bytes|allocator_selected" "$LOG" | tail -20
  echo
done

Active profile: t4
=== logs/smoke_t4_apr9.txt ===
  quantized_model_path: final_model.int6.ptz
val_tokens: 1046528
pre-quantization post-ema val_loss:6.78491669 val_bpb:2.66379936 eval_time:3595ms
Serialized model quantized+brotli: 16008201 bytes
Total submission size quantized+brotli: 16024795 bytes
quantized val_loss:6.78783732 val_bpb:2.66494602 eval_time:3760ms
quantized_sliding_window val_loss:6.78924600 val_bpb:2.66549907 eval_time:69441ms

=== logs/smoke_t4_apr20_attngate_mpttt.txt ===
  quantized_model_path: final_model.int6.ptz
val_tokens:1046528
pre-quantization post-ema val_loss:6.80454087 val_bpb:2.67150393 eval_time:3687ms
allocator_selected lambda:1e-07 wrapper:lzma_raw_b85_exec score:1.888943614e+00 model_bytes:14811218 code_bytes:25060 total_bytes:14836278 target_bytes:16000000
Serialized model quantized+brotli: 14811218 bytes  raw_torch:36616449
Total submission size quantized+brotli: 14836278 bytes
quantized val_loss:6.80475995 val_bpb:2.67158995 eval_time:3867ms
quan